In [1]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math
# 1. Set seed for Python's built-in random module (if used)
import random
SEED = 42
random.seed(SEED)

# 2. Set seed for NumPy
np.random.seed(SEED)

# 3. Set seed for PyTorch (CPU and all GPUs)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

# 4. Enforce strict CUDNN determinism
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# 5. Enforce deterministic PyTorch operations (throws error if non-deterministic operation is called)
torch.use_deterministic_algorithms(True, warn_only=True)

# 6. Set DataLoader generator (if using PyTorch DataLoaders with multiple workers)
g = torch.Generator()
g.manual_seed(SEED)

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
import numpy as np
import torch

def generate_pde_data_system_1(num_samples=1000, time_steps=50, grid_points=128):
    """
    Generates 1D Kuramoto-Sivashinsky Equation trajectories with strict
    value clipping, safety guards, and randomized initial conditions.
    """
    L = 2.0 * np.pi
    x = np.linspace(0, L, grid_points, endpoint=False)
    dx = L / grid_points
    t = np.linspace(0.0, 1.0, time_steps)
    dt = t[1] - t[0]

    data = []
    train_split = int(0.8 * num_samples)

    k = np.fft.rfftfreq(grid_points, d=dx) * 2.0 * np.pi
    dealias_mask = (np.abs(k) < (max(k) * 2.0 / 3.0))

    lin_op = (k**2) - (k**4)

    def compute_nonlinear_hat(u_curr):
        # CRITICAL FIX: Clip u_curr directly before squaring to prevent overflow
        u_safe = np.clip(u_curr, -20.0, 20.0)
        u_sq_hat = np.fft.rfft(u_safe ** 2) * dealias_mask
        # Fixed: Added wave number k and proper multiplication operator
        return -0.5 * (1j * k * u_sq_hat)

    for i in range(num_samples):
        is_rare_event = (np.random.rand() < 0.08) if i < train_split else (np.random.rand() < 0.25)

        # Added randomization for both normal and rare initial conditions
        if not is_rare_event:
            amp = np.random.uniform(0.15, 0.25)
            phase = np.random.uniform(0, 2 * np.pi)
            u0 = amp * np.cos(x + phase) * (1.0 + np.sin(x + phase))
        else:
            a1 = np.random.uniform(0.15, 0.25)
            a2 = np.random.uniform(0.08, 0.15)
            phase = np.random.uniform(0, 2 * np.pi)
            u0 = a1 * np.sin(2.0 * x + phase) + a2 * np.cos(3.0 * x - phase)

        u = np.zeros((time_steps, grid_points))
        u[0, :] = u0
        curr_u = u0.copy()

        substeps = 20
        dt_sub = dt / substeps

        for t_idx in range(1, time_steps):
            for _ in range(substeps):
                u_hat = np.fft.rfft(curr_u) * dealias_mask

                exp_half = np.exp(0.5 * dt_sub * lin_op)
                exp_full = np.exp(dt_sub * lin_op)

                n1 = compute_nonlinear_hat(curr_u)

                u_hat_2 = exp_half * u_hat + (0.5 * dt_sub) * n1
                u_2 = np.fft.irfft(u_hat_2, n=grid_points)
                n2 = compute_nonlinear_hat(u_2)

                u_hat_3 = exp_half * u_hat + (0.5 * dt_sub) * n2
                u_3 = np.fft.irfft(u_hat_3, n=grid_points)
                n3 = compute_nonlinear_hat(u_3)

                u_hat_4 = exp_full * u_hat + dt_sub * exp_half * n3
                u_4 = np.fft.irfft(u_hat_4, n=grid_points)
                n4 = compute_nonlinear_hat(u_4)

                u_hat_next = exp_full * u_hat + (dt_sub / 6.0) * (
                    exp_full * n1 + 2.0 * exp_half * n2 + 2.0 * exp_half * n3 + n4
                )
                u_hat_next *= dealias_mask

                curr_u = np.fft.irfft(u_hat_next, n=grid_points)

                # Fallback guard against numerical instability
                if np.isnan(curr_u).any() or np.isinf(curr_u).any():
                    curr_u = np.nan_to_num(curr_u, nan=0.0, posinf=10.0, neginf=-10.0)

            u[t_idx, :] = curr_u

        data.append(u)

    return torch.tensor(np.array(data), dtype=torch.float32)

In [3]:
def generate_pde_data_system_2(num_samples=1000, time_steps=50, grid_points=128):
    """
    Generates 1D Viscous Burgers' Equation trajectories (u_t + u * u_x = nu * u_xx).
    Features randomized kinematic viscosity (nu) and shock-formation profiles.
    """
    L = 2.0 * np.pi
    x = np.linspace(0, L, grid_points, endpoint=False)
    dx = L / grid_points
    t = np.linspace(0.0, 1.0, time_steps)
    dt = t[1] - t[0]

    data = []
    train_split = int(0.8 * num_samples)

    k = np.fft.rfftfreq(grid_points, d=dx) * 2.0 * np.pi
    dealias_mask = (np.abs(k) < (max(k) * 2.0 / 3.0))

    def rhs(u_curr, nu):
        """Right-hand side: -0.5 * (u^2)_x + nu * u_xx"""
        u_hat = np.fft.rfft(u_curr) * dealias_mask
        # Nonlinear term computed conservatively via flux derivative: (u^2)_x
        u_sq_hat = np.fft.rfft(u_curr ** 2) * dealias_mask
        
        # Fixed: Added missing '*' operators and wave number k for correct differentiation
        nonlinear = 0.5 * np.fft.irfft(1j * k * u_sq_hat, n=grid_points)
        diffusion = np.fft.irfft(-(k**2) * u_hat, n=grid_points)
        return -nonlinear + nu * diffusion

    for i in range(num_samples):
        is_rare_event = (np.random.rand() < 0.08) if i < train_split else (np.random.rand() < 0.25)

        if not is_rare_event:
            nu = np.random.uniform(0.01, 0.03)
            u0 = np.sin(x) + 0.5 * np.sin(2.0 * x)
        else:
            nu = np.random.uniform(0.002, 0.008)  # Very low viscosity = aggressive shocks
            u0 = -np.sin(x) * 1.5

        u = np.zeros((time_steps, grid_points))
        u[0, :] = u0
        curr_u = u0.copy()

        substeps = 10
        dt_sub = dt / substeps

        for t_idx in range(1, time_steps):
            for _ in range(substeps):
                k1 = rhs(curr_u, nu)
                k2 = rhs(curr_u + 0.5 * dt_sub * k1, nu)
                k3 = rhs(curr_u + 0.5 * dt_sub * k2, nu)
                k4 = rhs(curr_u + dt_sub * k3, nu)
                
                curr_u = curr_u + (dt_sub / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)

            u[t_idx, :] = curr_u

        data.append(u)

    return torch.tensor(np.array(data), dtype=torch.float32)

In [4]:
def generate_pde_data_system_3(num_samples=1000, time_steps=50, grid_points=128):
    """
    Generates 1D Korteweg-de Vries (KdV) Equation trajectories (u_t + 6u*u_x + delta^2*u_xxx = 0).
    Features dispersive soliton dynamics and randomized dispersion coefficients.
    """
    L = 2.0 * np.pi
    x = np.linspace(0, L, grid_points, endpoint=False)
    dx = L / grid_points
    t = np.linspace(0.0, 0.5, time_steps)
    dt = t[1] - t[0]

    data = []
    train_split = int(0.8 * num_samples)

    k = np.fft.rfftfreq(grid_points, d=dx) * 2.0 * np.pi
    dealias_mask = (np.abs(k) < (max(k) * 2.0 / 3.0))

    def rhs(u_curr, delta_sq):
        """Right-hand side: -6 * u * u_x - delta^2 * u_xxx"""
        u_hat = np.fft.rfft(u_curr) * dealias_mask
        u_sq_hat = np.fft.rfft(u_curr ** 2) * dealias_mask

        # Fixed: Added wave numbers k and k^3 for correct pseudo-spectral derivatives
        nonlinear = 3.0 * np.fft.irfft(1j * k * u_sq_hat, n=grid_points) # 6 * 0.5 = 3
        dispersion = delta_sq * np.fft.irfft(-1j * (k**3) * u_hat, n=grid_points)
        return -nonlinear - dispersion

    for i in range(num_samples):
        is_rare_event = (np.random.rand() < 0.08) if i < train_split else (np.random.rand() < 0.25)

        if not is_rare_event:
            delta_sq = np.random.uniform(0.001, 0.003)
            c1, c2 = np.random.uniform(0.8, 1.2), np.random.uniform(0.4, 0.8)
            u0 = 3.0 * c1 * np.cosh(0.5 * np.sqrt(c1) * x)**(-2)
        else:
            delta_sq = np.random.uniform(0.004, 0.008) # Multi-soliton high dispersion
            u0 = 3.0 * np.cosh(0.5 * x)**(-2) + 2.0 * np.cosh(0.4 * (x - np.pi))**(-2)

        u = np.zeros((time_steps, grid_points))
        u[0, :] = u0
        curr_u = u0.copy()

        substeps = 15
        dt_sub = dt / substeps

        for t_idx in range(1, time_steps):
            for _ in range(substeps):
                k1 = rhs(curr_u, delta_sq)
                k2 = rhs(curr_u + 0.5 * dt_sub * k1, delta_sq)
                k3 = rhs(curr_u + 0.5 * dt_sub * k2, delta_sq)
                k4 = rhs(curr_u + dt_sub * k3, delta_sq)
                
                curr_u = curr_u + (dt_sub / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)

            u[t_idx, :] = curr_u

        data.append(u)

    return torch.tensor(np.array(data), dtype=torch.float32)

In [5]:
def generate_pde_data_system_4(num_samples=1000, time_steps=50, grid_points=128):
    """Generates 1D Advection-Diffusion Equation trajectories (u_t + v * u_x = D * u_xx).

    Features randomized advection velocity (v) and diffusion coefficient (D) per sample.
    Mathematically stable, linear, and completely overflow-free.

    Returns a single tensor of shape (num_samples, time_steps, grid_points).
    First 80% (0..799) -> Standard training pool.
    Last 20%  (800..999) -> Test pool with aggressive advection and steep profiles.
    """
    L = 2.0 * np.pi
    x = np.linspace(0, L, grid_points, endpoint=False)
    dx = L / grid_points
    t = np.linspace(0.0, 1.0, time_steps)
    dt = t[1] - t[0]

    data = []
    train_split = int(0.8 * num_samples)

    # Wave number array for pseudo-spectral derivatives
    k = np.fft.rfftfreq(grid_points, d=dx) * 2.0 * np.pi
    dealias_mask = (np.abs(k) < (max(k) * 2.0 / 3.0))

    def rhs(u_curr, v, D):
        """Right-hand side: -v * u_x + D * u_xx"""
        u_hat = np.fft.rfft(u_curr) * dealias_mask
        # Fixed: Added wave number k and correct multiplication
        u_x = np.fft.irfft(1j * k * u_hat, n=grid_points)
        u_xx = np.fft.irfft(-(k**2) * u_hat, n=grid_points)
        return -v * u_x + D * u_xx

    for i in range(num_samples):
        is_rare_event = (np.random.rand() < 0.08) if i < train_split else (np.random.rand() < 0.25)

        # Random physical parameters chosen per sample
        if not is_rare_event:
            v = np.random.uniform(0.5, 1.5)  # Moderate advection speed
            D = np.random.uniform(0.02, 0.08) # Standard diffusion
            u0 = np.sin(x) + 0.5 * np.cos(2.0 * x)
        else:
            v = np.random.uniform(1.8, 3.0)  # Fast, aggressive transport speed
            D = np.random.uniform(0.005, 0.02) # Low diffusion (sharp wave retention)
            u0 = np.sin(3.0 * x) * np.exp(-((x - np.pi)**2) / 2.0)

        u = np.zeros((time_steps, grid_points))
        u[0, :] = u0
        curr_u = u0.copy()

        # Stable RK4 integration
        substeps = 10
        dt_sub = dt / substeps

        for t_idx in range(1, time_steps):
            for _ in range(substeps):
                k1 = rhs(curr_u, v, D)
                k2 = rhs(curr_u + 0.5 * dt_sub * k1, v, D)
                k3 = rhs(curr_u + 0.5 * dt_sub * k2, v, D)
                k4 = rhs(curr_u + dt_sub * k3, v, D)

                curr_u = curr_u + (dt_sub / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)

            u[t_idx, :] = curr_u

        data.append(u)

    return torch.tensor(np.array(data), dtype=torch.float32)

In [6]:
def prepare_training_pairs(pde_trajectories: torch.Tensor, history_steps: int = 10):
    X, Y = [], []
    num_samples, total_time, grid_size = pde_trajectories.shape

    for t in range(history_steps, total_time):
        x_seq = pde_trajectories[:, t - history_steps : t, :].permute(0, 2, 1)
        y_seq = pde_trajectories[:, t : t + 1, :].permute(0, 2, 1)
        X.append(x_seq)
        Y.append(y_seq)

    X = torch.cat(X, dim=0)
    Y = torch.cat(Y, dim=0)
    return X, Y

In [7]:
def plot_trajectory_summary(sample_idx, title_prefix, mse_val, mae_val, r2_val,
                            ground_truth_np, predicted_np, history_steps, grid_points):
    actual_sample = ground_truth_np[sample_idx]
    predicted_sample = predicted_np[sample_idx]

    fig = plt.figure(figsize=(15, 10))
    fig.suptitle(
        f"{title_prefix} (Sample #{sample_idx + 1})\n"
        f"MSE: {mse_val:.6f} | MAE: {mae_val:.6f} | R² Score: {r2_val:.4f}",
        fontsize=14, fontweight='bold'
    )

    # 1. Heatmap: Ground Truth
    ax1 = fig.add_subplot(2, 2, 1)
    im1 = ax1.imshow(actual_sample, aspect='auto', cmap='jet', origin='lower')
    ax1.set_title("Actual Ground Truth (Space-Time)")
    ax1.set_xlabel("Spatial Grid (x)")
    ax1.set_ylabel("Time Step (t)")
    ax1.axhline(y=history_steps - 0.5, color='black', linestyle='--', label=f'End of initial {history_steps} steps')
    fig.colorbar(im1, ax=ax1)

    # 2. Heatmap: FNO Auto-regressive Prediction
    ax2 = fig.add_subplot(2, 2, 2)
    im2 = ax2.imshow(predicted_sample, aspect='auto', cmap='jet', origin='lower')
    ax2.set_title("FNO Prediction (Auto-regressive)")
    ax2.set_xlabel("Spatial Grid (x)")
    ax2.set_ylabel("Time Step (t)")
    ax2.axhline(y=history_steps - 0.5, color='black', linestyle='--', label=f'End of initial {history_steps} steps')
    fig.colorbar(im2, ax=ax2)

    # 3. Absolute Error Heatmap
    ax3 = fig.add_subplot(2, 2, 3)
    error_sample = np.abs(actual_sample - predicted_sample)
    im3 = ax3.imshow(error_sample, aspect='auto', cmap='Reds', origin='lower')
    ax3.set_title("Absolute Error |Actual - Predicted|")
    ax3.set_xlabel("Spatial Grid (x)")
    ax3.set_ylabel("Time Step (t)")
    fig.colorbar(im3, ax=ax3)

    # 4. Snapshot profiles at specific future time steps
    ax4 = fig.add_subplot(2, 2, 4)
    x_axis = np.linspace(0, 1, grid_points)
    time_snapshots = [history_steps + 2, 25, 45]  # Time steps to view cross-sections

    for t_snap in time_snapshots:
        ax4.plot(x_axis, actual_sample[t_snap], label=f"Actual (t={t_snap})", linestyle='--')
        ax4.plot(x_axis, predicted_sample[t_snap], label=f"Pred (t={t_snap})")

    ax4.set_title("Spatial Profile Snapshots")
    ax4.set_xlabel("Spatial Grid (x)")
    ax4.set_ylabel("u(x, t)")
    ax4.legend(loc='upper right', fontsize='small')
    ax4.grid(True)

    plt.tight_layout()
    plt.show()


In [8]:
class SpectralConv1d(nn.Module):
    """
    1D Fourier Spectral Convolution Layer for accurate PDE operator learning.
    """
    def __init__(self, in_channels: int, out_channels: int, modes1: int):
        super(SpectralConv1d, self).__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.modes1 = modes1

        scale = 1.0 / (in_channels * out_channels)
        self.weights1 = nn.Parameter(
            scale * torch.randn(in_channels, out_channels, self.modes1, dtype=torch.cfloat)
        )

    def forward(self, x):
        batchsize = x.shape[0]
        x_ft = torch.fft.rfft(x)

        out_ft = torch.zeros(
            batchsize, self.out_channels, x.size(-1) // 2 + 1,
            device=x.device, dtype=torch.cfloat
        )
        out_ft[:, :, :self.modes1] = torch.einsum(
            "bix,iox->box", x_ft[:, :, :self.modes1], self.weights1
        )

        return torch.fft.irfft(out_ft, n=x.size(-1))


# =====================================================================
# 2. Stabilized Gated Spatial Mixer Block
# =====================================================================
class MambaMixerBlock1d(nn.Module):
    """
    Gated Mamba mixer block optimized with Pre-Norm and scaled residuals
    to prevent autoregressive drift and ensure a strong positive R² score.
    """
    def __init__(self, d_model: int, expand_factor: int = 2):
        super(MambaMixerBlock1d, self).__init__()
        self.d_model = d_model
        self.inner_dim = d_model * expand_factor

        self.norm = nn.LayerNorm(d_model)
        self.in_proj = nn.Linear(d_model, self.inner_dim * 2)

        self.conv1d = nn.Conv1d(
            in_channels=self.inner_dim,
            out_channels=self.inner_dim,
            kernel_size=5,
            padding=2,
            groups=self.inner_dim
        )

        self.fc1 = nn.Linear(self.inner_dim, self.inner_dim)
        self.fc2 = nn.Linear(self.inner_dim, d_model)
        self.gate = nn.Linear(self.inner_dim, d_model)

        # Scaled residual parameter for rollout stability
        self.res_scale = nn.Parameter(torch.ones(1) * 0.1)

    def forward(self, x):
        residual = x
        x_norm = self.norm(x)

        xz = self.in_proj(x_norm)
        x_b, z = torch.chunk(xz, 2, dim=-1)

        x_b = x_b.permute(0, 2, 1)
        x_b = F.gelu(self.conv1d(x_b))
        x_b = x_b.permute(0, 2, 1)

        h = F.gelu(self.fc1(x_b))
        out = self.fc2(h) * torch.sigmoid(self.gate(z))

        return residual + self.res_scale * out


# =====================================================================
# 3. High-Accuracy Fourier-Mamba Operator Network
# =====================================================================
class FourierMambaOperator1d(nn.Module):
    """
    Hybrid Fourier-Mamba Operator Network for 1D PDE Surrogate Modeling.
    """
    def __init__(
        self,
        history_steps: int = 10,
        width: int = 64,
        modes: int = 16,
        num_layers: int = 4,
        dropout: float = 0.05
    ):
        super(FourierMambaOperator1d, self).__init__()
        self.history_steps = history_steps
        self.width = width
        self.modes = modes

        self.fc0 = nn.Linear(self.history_steps + 1, self.width)
        self.spectral_conv = SpectralConv1d(self.width, self.width, self.modes)
        self.w_conv = nn.Conv1d(self.width, self.width, 1)

        self.mamba_layers = nn.ModuleList([
            MambaMixerBlock1d(d_model=self.width, expand_factor=2)
            for _ in range(num_layers)
        ])

        self.dropout = nn.Dropout(dropout)
        self.norm_out = nn.LayerNorm(self.width)
        self.fc1 = nn.Linear(self.width, 128)
        self.fc2 = nn.Linear(128, 1)

    def forward(self, x):
        batchsize, size_x, _ = x.shape
        grid = self.get_grid(x.shape, x.device)
        x = torch.cat((x, grid), dim=-1)

        x = self.fc0(x)
        x_chan = x.permute(0, 2, 1)
        x_spec = F.gelu(self.spectral_conv(x_chan) + self.w_conv(x_chan))
        x = x_spec.permute(0, 2, 1)

        for layer in self.mamba_layers:
            x = layer(x)
            x = self.dropout(x)

        x = self.norm_out(x)
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.fc2(x)

        return x

    def get_grid(self, shape, device):
        batchsize, size_x = shape[0], shape[1]
        gridx = torch.linspace(0, 1, size_x, device=device, dtype=torch.float32)
        gridx = gridx.reshape(1, size_x, 1).repeat([batchsize, 1, 1])
        return gridx

In [9]:
def evaluate_system(system_name, generate_data_fn):
    print(f"\n=======================================================")
    print(f"       EVALUATING: {system_name.upper()}               ")
    print(f"=======================================================")

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    num_pdes = 1000
    time_steps = 50
    grid_points = 128
    history_steps = 10

    # Fixed: Correct multiplication syntax
    training_amount = int(0.8 * num_pdes)

    # Generate synthetic PDE solutions using the specific system function
    all_pde_data = generate_data_fn(num_pdes, time_steps, grid_points)

    train_data = all_pde_data[:training_amount]
    test_data = all_pde_data[training_amount:]

    X_train, Y_train = prepare_training_pairs(train_data, history_steps=history_steps)
    train_dataset = torch.utils.data.TensorDataset(X_train, Y_train)
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)

    # Initialize Stabilized Fourier-Mamba Operator
    model = FourierMambaOperator1d(history_steps=history_steps, modes=16, width=64, num_layers=3).to(device)

    # Use AdamW + Cosine Annealing for stable rollout learning
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=40, eta_min=1e-5)
    criterion = nn.MSELoss()

    # Train model
    epochs = 50
    model.train()
    print(f"Training Fourier-Mamba Operator Model for {system_name}...")
    for epoch in range(1, epochs + 1):
        total_loss = 0.0
        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            out = model(x_batch)
            loss = criterion(out, y_batch)
            loss.backward()

            # Gradient clipping prevents exploding gradients during autoregressive rollout
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()
            total_loss += loss.item()

        scheduler.step()
        avg_loss = total_loss / len(train_loader)
        if epoch % 10 == 0 or epoch == 1:
            print(f"Epoch [{epoch}/{epochs}] - Loss: {avg_loss:.6f}")

    # Rollout on test set
    model.eval()
    rollout_steps = time_steps - history_steps
    current_history = test_data[:, :history_steps, :].permute(0, 2, 1).to(device)
    predicted_frames = []

    with torch.no_grad():
        for _ in range(rollout_steps):
            pred_next = model(current_history)
            predicted_frames.append(pred_next.permute(0, 2, 1))
            current_history = torch.cat([current_history[:, :, 1:], pred_next], dim=-1)

    # Combine initial history frames + auto-regressive predictions
    predictions_only = torch.cat(predicted_frames, dim=1).cpu()
    initial_frames = test_data[:, :history_steps, :]
    full_predicted_trajectories = torch.cat([initial_frames, predictions_only], dim=1)

    # Convert tensors to NumPy arrays for evaluation
    ground_truth_np = test_data.numpy()
    predicted_np = full_predicted_trajectories.numpy()

    # Focus evaluation on the autoregressive rollout horizon
    gt_rollout = ground_truth_np[:, history_steps:, :]
    pred_rollout = predicted_np[:, history_steps:, :]

    # =====================================================================
    # Compute Overall Test Metrics
    # =====================================================================
    rollout_mse = np.mean((pred_rollout - gt_rollout) ** 2)
    rollout_mae = np.mean(np.abs(pred_rollout - gt_rollout))

    l2_norms = np.linalg.norm(pred_rollout - gt_rollout, axis=(1, 2))
    gt_norms = np.linalg.norm(gt_rollout, axis=(1, 2))
    relative_l2_error = np.mean(l2_norms / (gt_norms + 1e-8))

    # Fixed: Added missing subtraction operator
    ss_res_total = np.sum((gt_rollout - pred_rollout) ** 2)
    ss_tot_total = np.sum((gt_rollout - np.mean(gt_rollout)) ** 2)
    rollout_r2 = 1.0 - (ss_res_total / (ss_tot_total + 1e-8))

    half_horizon = rollout_steps // 2
    long_horizon_mse = np.mean((pred_rollout[:, half_horizon:, :] - gt_rollout[:, half_horizon:, :]) ** 2)

    # =====================================================================
    # Compute Metrics Per Trajectory (Best vs Worst & Percentage Distributions)
    # =====================================================================
    mse_per_sample = np.mean((ground_truth_np - predicted_np) ** 2, axis=(1, 2))
    mae_per_sample = np.mean(np.abs(ground_truth_np - predicted_np), axis=(1, 2))

    r2_per_sample = []
    for i in range(len(ground_truth_np)):
        y_true = ground_truth_np[i].flatten()
        y_pred = predicted_np[i].flatten()
        ss_res = np.sum((y_true - y_pred) ** 2)
        ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
        # Fixed: Added missing subtraction operator
        r2 = 1.0 - (ss_res / (ss_tot + 1e-8))
        r2_per_sample.append(r2)
    r2_per_sample = np.array(r2_per_sample)

    # Calculate percentage metrics
    pct_r2_ge_90 = np.mean(r2_per_sample >= 0.9) * 100.0
    pct_r2_lt_0 = np.mean(r2_per_sample < 0.0) * 100.0  # Percentage of R² < 0

    best_idx = int(np.argmin(mse_per_sample))
    worst_idx = int(np.argmax(mse_per_sample))

    # Print Full Test Evaluation Report
    print(f"\n=======================================================")
    print(f"       TEST EVALUATION REPORT: {system_name.upper()}     ")
    print(f"=======================================================")
    print(f"  • Relative L2 Error         : {relative_l2_error:.6f}")
    print(f"  • Rollout MSE               : {rollout_mse:.6f}")
    print(f"  • Rollout MAE               : {rollout_mae:.6f}")
    print(f"  • Rollout R² Score          : {rollout_r2:.6f}")
    print(f"  • Long-Horizon MSE          : {long_horizon_mse:.6f}")
    print(f"  • Trajectories with R² >= 0.9: {pct_r2_ge_90:.2f}%")
    print(f"  • Trajectories with R² < 0  : {pct_r2_lt_0:.2f}%")
    print(f"=======================================================\n")

    print(f"Best Test Trajectory Index:  #{best_idx + 1} | MSE: {mse_per_sample[best_idx]:.6f} | R²: {r2_per_sample[best_idx]:.4f}")
    print(f"Worst Test Trajectory Index: #{worst_idx + 1} | MSE: {mse_per_sample[worst_idx]:.6f} | R²: {r2_per_sample[worst_idx]:.4f}\n")


def main():
    # Map your systems to their respective data generation functions
    systems = {
        "System 1": generate_pde_data_system_1,
        "System 2": generate_pde_data_system_2,
        "System 3": generate_pde_data_system_3,
        "System 4": generate_pde_data_system_4
    }

    for name, gen_fn in systems.items():
        evaluate_system(name, gen_fn)


if __name__ == "__main__":
    main()


       EVALUATING: SYSTEM 1               
Using device: cuda
Training Fourier-Mamba Operator Model for System 1...
Epoch [1/50] - Loss: 0.000504
Epoch [10/50] - Loss: 0.000010
Epoch [20/50] - Loss: 0.000003
Epoch [30/50] - Loss: 0.000001
Epoch [40/50] - Loss: 0.000001
Epoch [50/50] - Loss: 0.000001

       TEST EVALUATION REPORT: SYSTEM 1     
  • Relative L2 Error         : 0.519745
  • Rollout MSE               : 0.000024
  • Rollout MAE               : 0.004037
  • Rollout R² Score          : 0.998383
  • Long-Horizon MSE          : 0.000033
  • Trajectories with R² >= 0.9: 100.00%
  • Trajectories with R² < 0  : 0.00%

Best Test Trajectory Index:  #17 | MSE: 0.000008 | R²: 0.9996
Worst Test Trajectory Index: #103 | MSE: 0.000037 | R²: 0.9988


       EVALUATING: SYSTEM 2               
Using device: cuda
Training Fourier-Mamba Operator Model for System 2...
Epoch [1/50] - Loss: 0.003541
Epoch [10/50] - Loss: 0.000109
Epoch [20/50] - Loss: 0.000050
Epoch [30/50] - Loss: 0.000029
E